In [1]:
import os
import argparse
import random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from softprompt_experiments.models.softprompt import SoftPrompt
from softprompt_experiments.utils import log_json

from peft import PeftModel
import pandas as pd
from tqdm import tqdm
import evaluate
import json


SAVE_DIRECTORY = "/home/pkongsomjit/Projects/softprompt_experiments/datasets/generalization_datasets"
# Parse all the arguments into Variables
PROPORTION_FOLDER = f"{int(100*1.0)}_percent"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
# LORA_DIR = "/home/pkongsomjit/Projects/softprompt_experiments/new_mapper_lora_weights"
LORA_DIR = "/home/pkongsomjit/Projects/softprompt_experiments/shared/mapper_lora_weights/General-DoD1/meta-llama/Llama-3.1-8B-Instruct"

# TRAINING_STATS_PATH = args.training_stats_path
SEED = 42
DO_SAMPLE = True
# OUTPUT_JSON = args.output_json

# Set the Seed for this experiment
torch.manual_seed(SEED)
random.seed(SEED)

# Determine DEVICE and DTYPE
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

# Load Training Stats
# TRAINING_STATS_DF = pd.read_csv(TRAINING_STATS_PATH, index_col='task_name')

# ┌───────────────────────────────────────────────┐
# │                 LORA MODEL PREP               │
# └───────────────────────────────────────────────┘
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model {MODEL_NAME}...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map=DEVICE)
word_embeddings = base_model.get_input_embeddings()

print(f"Loading LoRA adapters from {LORA_DIR}...")
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

# ┌───────────────────────────────────────────────┐
# │                 INFERENCE LOOP                │
# └───────────────────────────────────────────────┘
print("\n" + "="*80)
print("\t\t\tGENERATION RESULTS")
print("="*80 + "\n")

results_data = []

dataset_dirs = []
for entry in os.scandir(SAVE_DIRECTORY):
    if entry.is_dir():  # Check if the entry is a directory
        if "dataset_" in entry.name:
            dataset_dirs.append(entry.path)

num_datasets = len(dataset_dirs)
if num_datasets > 0:
    print(f"\nFound ({num_datasets}) datasets in directory")
else:
    raise ValueError("path to directory has no datasets")

Loading base model meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading LoRA adapters from /home/pkongsomjit/Projects/softprompt_experiments/shared/mapper_lora_weights/General-DoD1/meta-llama/Llama-3.1-8B-Instruct...

			GENERATION RESULTS


Found (2) datasets in directory


In [2]:
prefill_text = "If"

softprompts = []

for dataset_dir in tqdm(dataset_dirs):
    print(f"For softprompt in {dataset_dir}")
    # hardprompt = torch.load(
    #     os.path.join(dataset_dir,'dataset.pt'),
    #     weights_only=False
    # )['hardprompt']
    results = {}
    # results['hardprompt'] = hardprompt
    softprompt = SoftPrompt(
        model=model, 
        tokenizer=tokenizer, 
        word_embeddings=None, 
        path_to_model=os.path.join(dataset_dir,'softprompt.pt')
    )
    softprompts.append(softprompt.forward()) 

100%|██████████| 2/2 [00:00<00:00, 94.15it/s]

For softprompt in /home/pkongsomjit/Projects/softprompt_experiments/datasets/generalization_datasets/dataset_human_or_ai
For softprompt in /home/pkongsomjit/Projects/softprompt_experiments/datasets/generalization_datasets/dataset_deceptive_op_spam


In [3]:
def verbalize_softprompt(softprompt, do_beam=True, num_beams=3):
    model.eval()
    with torch.no_grad():
        # Just softprompt
        soft_attn_mask = torch.ones(softprompt.shape[:2], dtype=torch.long, device=DEVICE)
    
        # Prefill
        tokenized_input = tokenizer(prefill_text, return_tensors='pt',add_special_tokens=False).to(DEVICE)
        input_embeds = word_embeddings(tokenized_input['input_ids']).to(DEVICE).to(DTYPE)
        input_attn_mask = tokenized_input['attention_mask']
        
        # Soft + prefill
        full_embeds = torch.cat([softprompt, input_embeds], dim=1)
        full_attn_mask = torch.cat([soft_attn_mask, input_attn_mask], dim=1)
    
        # Random control
        random_embeds = word_embeddings(torch.randint(
            0, word_embeddings.num_embeddings,(softprompt.shape[1],), dtype=torch.long
        ).to(model.device)).to(DEVICE).to(DTYPE).unsqueeze(0)
    
        # Random + prefill
        random_full_embeds = torch.cat([random_embeds, input_embeds], dim=1)

        if do_beam:
            NUM_BEAMS = num_beams
    
            outputs = model.generate(
                inputs_embeds=softprompt,
                attention_mask=soft_attn_mask,
                max_new_tokens=300,
                do_sample=False,
                num_beams=NUM_BEAMS,
                num_return_sequences=NUM_BEAMS,
                early_stopping=True,
                pad_token_id=tokenizer.eos_token_id,
            )
            
            pred_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            for i, text in enumerate(pred_texts):
                print(f"Beam {i}:")
                print(f"<v begin>{text.strip()}</v end>\n")
        else:
    
            # SOFTPROMPT TRIAL-----------------------------------------------------------------
            outputs = model.generate(
                inputs_embeds=softprompt,
                attention_mask=soft_attn_mask,
                max_new_tokens=300,
                do_sample=DO_SAMPLE,
                temperature=0.3 if DO_SAMPLE else None,
                top_p=0.9 if DO_SAMPLE else None,
                pad_token_id=tokenizer.eos_token_id
            )
            # Decode the batched outputs
            pred_texts_vanilla = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            pred_texts_vanilla = [text.strip() for text in pred_texts_vanilla]
        
            print(f"Verbalization using: soft embeds:\n<v begin>{pred_texts_vanilla[0]}</v end>\n")
        
        # # SOFT + PREFILL TRIAL-----------------------------------------------------------------
        # outputs = model.generate(
        #     inputs_embeds=full_embeds,
        #     attention_mask=full_attn_mask,
        #     max_new_tokens=300,
        #     do_sample=DO_SAMPLE,
        #     temperature=0.7 if DO_SAMPLE else None,
        #     top_p=0.9 if DO_SAMPLE else None,
        #     pad_token_id=tokenizer.eos_token_id
        # )
        
        # # Decode the batched outputs
        # pred_texts_prefilled = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        # pred_texts_prefilled = [text.strip() for text in pred_texts_prefilled]
    
        # print(f"Verbalization using: soft embeds + prefill(\"{prefill_text}\"):\n{prefill_text}<v begin>{pred_texts_prefilled[0]}</v end>\n")


In [4]:
verbalize_softprompt(softprompts[0], num_beams=1)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Beam 0:
<v begin>In this task, you're given a short text which is either a statement or a question. You're expected to generate a response to the input. The response should be relevant to the input and should not contain any irrelevant information.</v end>



In [5]:
verbalize_softprompt(softprompts[0], num_beams=2)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Beam 0:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two categories: 1) if the post is written by a machine, 2) if the post is written by a human.</v end>

Beam 1:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two categories: 1) if the post is written by a machine, 2) if the post is written by humans.</v end>



In [6]:
verbalize_softprompt(softprompts[0], num_beams=3)

Beam 0:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two categories: 1) if the post is written by a machine, 2) if the post is written by a human.</v end>

Beam 1:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two categories: 1) if the post is written by a machine or 2) if the post is written by a human.</v end>

Beam 2:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two categories: 1) if the post is written by a machine, 2) if the post is written by humans.</v end>



In [7]:
verbalize_softprompt(softprompts[0], num_beams=5)

Beam 0:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two classes: 1) if the post is written by a machine, 2) if the post is written by a human. Note that the URLs in the text have been replaced with [Link].</v end>

Beam 1:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two classes: 1) if the post is written by a machine, 2) if the post is written by a human.</v end>

Beam 2:
<v begin>In this task, you're given a short text which is the body of a post in a social network. Your task is to classify the post into two classes: 1) if the post is written by a machine, 2) if the post is written by a human.</v end>

Beam 3:
<v begin>In this task, you're given a short text which is the body of a post in a social media platform. Your task is to classify the post into two classes: 1) if the post is writt

In [8]:
verbalize_softprompt(softprompts[0], num_beams=10)

Beam 0:
<v begin>In this task, you're given a short conversation between an automated system and a user looking for suggestions for restaurants in Cambridge. The user may provide some criteria for the type of restaurant they want such as price range, cuisine, etc. Your job is to generate a response to the user's question based on the context of the conversation.</v end>

Beam 1:
<v begin>In this task, you're given a short conversation between an automated system and a user looking for suggestions for restaurants in Cambridge. The user may provide some criteria for the type of restaurant they want such as price range, cuisine, etc. Given such a dialogue, answer the user's question with a suggestion for a restaurant in Cambridge.</v end>

Beam 2:
<v begin>In this task, you're given a short conversation between an automated system and a user looking for suggestions for restaurants in Cambridge. The user may provide some criteria for the type of restaurant they want such as price range, cu

In [9]:
verbalize_softprompt(softprompts[0], num_beams=20)

Beam 0:
<v begin>We would like you to classify each of the following sentences into either (1) 'generated' or (2) 'engaging-freedom'. A sentence is considered 'generated' if it could have been produced by a machine. A sentence is considered 'engaging-freedom' if it could not have been produced by a machine.</v end>

Beam 1:
<v begin>We would like you to classify each of the following sentences into either (1) 'generated' or (2) 'engaging-freedom'. A sentence is considered 'generated' if it could have been produced by a machine. A sentence is considered 'engaging-freedom' if it is creative and could not have been produced by a machine.</v end>

Beam 2:
<v begin>We would like you to classify each of the following sentences into either (1) 'generated' or (2) 'engaging-freedom'. A sentence is considered 'generated' if it could have been produced by a machine. On the other hand, a sentence is considered 'engaging-freedom' if it could not have been produced by a machine.</v end>

Beam 3:
<v 

In [10]:
[verbalize_softprompt(softprompts[0], do_beam=False) for _ in range(5)]

Verbalization using: soft embeds:
<v begin>In this task, you're given a short text which is either a statement or a question. You're expected to generate a response to the input. The response should be relevant to the input and should not contain any irrelevant information.</v end>

Verbalization using: soft embeds:
<v begin>In this task, you're given a short text which is either (1) a statement, (2) a question, or (3) an answer. Your task is to classify the given text into statement, question, or answer.</v end>

Verbalization using: soft embeds:
<v begin>In this task, you're given a short text which is either a statement or a question. If it's a statement, you're expected to generate a most likely context for the given statement. If it's a question, you're expected to generate a most likely context for the given question. The context should contain the answer to the question or should support the statement.</v end>

Verbalization using: soft embeds:
<v begin>In this task, you're give

[None, None, None, None, None]

In [11]:
verbalize_softprompt(softprompts[1], do_beam=False)

Verbalization using: soft embeds:
<v begin>In this task, you're given a context paragraph, a question about the context, and an answer to that question. Your job is to classify the given answer as "truthful" or "deceptive" based on commonsense reasoning about social situations. Answer "truthful" if the answer is correct, otherwise answer "deceptive".</v end>



In [12]:
[verbalize_softprompt(softprompts[1], do_beam=False) for _ in range(5)]

Verbalization using: soft embeds:
<v begin>In this task, you're given a context paragraph, a question about the context, and an answer to that question. Your task is to classify the given answer as "truthful" or "deceptive" based on common sense and reasoning about social situations.</v end>

Verbalization using: soft embeds:
<v begin>In this task, you're given a context paragraph, a question about the context, and an answer to that question. Your task is to classify the answer as "truthful" or "deceptive" based on common sense and reasoning about social situations. Note that the answer may not be completely incorrect, but it may not fully answer the question or may not be the best answer.</v end>

Verbalization using: soft embeds:
<v begin>In this task, you're given a context paragraph, a sentence, and a yes/no question. Your task is to classify the given sentence as truthy or falsy based on the context paragraph. If the sentence is truthy, the sentence describes a valid event that ca

[None, None, None, None, None]

In [13]:
verbalize_softprompt(softprompts[1], num_beams=5)

Beam 0:
<v begin>In this task, you're given a context, a question, and an answer. Your task is to classify the given answer as "truthful" or "deceptive" based on commonsense reasoning about social situations. Answer "truthful" if the answer is correct, otherwise answer "deceptive".</v end>

Beam 1:
<v begin>In this task, you're given a context, a question, and an answer. Your task is to classify the given answer as "truthful" or "deceptive" based on commonsense reasoning about social situations. Answer "truthful" if the given answer is correct, otherwise answer "deceptive".</v end>

Beam 2:
<v begin>In this task, you're given a context, a question, and an answer. Your task is to classify the given answer as "truthful" or "deceptive" based on commonsense reasoning about social situations.</v end>

Beam 3:
<v begin>In this task, you're given a context, a question, and an answer. Your task is to classify the given answer as "truthful" or "deceptive" based on the context and the question.<

In [14]:
verbalize_softprompt(softprompts[1], num_beams=10)

Beam 0:
<v begin>In this task, you're given a context, a question, and an answer. Your task is to classify the given answer as "truthful" or "deceptive" based on commonsense reasoning about social situations. Answer "truthful" if the answer is correct, otherwise answer "deceptive".</v end>

Beam 1:
<v begin>In this task, you're given a context, a question, and an answer. Your task is to classify the given answer as "truthful" or "deceptive" based on commonsense reasoning about social situations. Answer "truthful" if the answer correctly answers the question and is truthful, otherwise answer "deceptive".</v end>

Beam 2:
<v begin>In this task, you're given a context, a question, and an answer. Your task is to classify the given answer as "truthful" or "deceptive" based on commonsense reasoning about social situations. Answer "truthful" if the answer correctly answers the question and is reasonable given the context, otherwise answer "deceptive".</v end>

Beam 3:
<v begin>In this task, y

In [15]:
composite = torch.cat([softprompts[0][:,:10], softprompts[1][:,10:]])